# ILC method

After the first two steps:
1. Analythical work on g($\nu$),
2. Simultaions of tSZ + CMB + noise signals,

I will start implementing the ILC method.

The code must
1. Compute the correlation matrix: $C_{ij} = \frac{1}{N_P} \Sigma_{p'} \left( d_i(p') \cdot d_j(p') \right)$, (i.e. i,j = 1, ..., 20)
2. Invert $\underline{\underline{C}} \rightarrow \underline{\underline{C^{-1}}}$
3. Compute the weights: $\omega_i = \frac{\underline{\underline{C^{-1}}} \cdot \underline{g}}{\underline{g^\intercal} \cdot \underline{\underline{C^{-1}}} \cdot \underline{g}}$
4. Apply the weights ($\omega_i$) to the data ($d_i$)
5. Get the output: $\hat{y} = \Sigma_i \left( \omega_i \cdot d_i (p) \right) $

---
Emprar vector a = [1,..,1] per provar de fer ILC NOMÉS sobre CMB
---
---

In [ ]:
### IMPORT PACKAGES ###

import healpy as hp
import numpy as np
import pandas as pd
import scipy as sp
import numba as nb
import matplotlib.pyplot as plt

import gc
from joblib import Parallel, delayed
from numba import njit, prange
from scipy.interpolate import *

from astropy.wcs import WCS
from reproject import reproject_from_healpix

In [ ]:
### MEMORY MANAGEMENT ###

import gc # Garbage Collector

# Before starting a new run, clear previous big variables if they exist
if 'data_cube' in locals():
    del data_cube
if 'alms_list' in locals():
    del alms_list
if 'y_hat_harmonic' in locals():
    del y_hat_harmonic

gc.collect() # Manually trigger memory cleanup

In [ ]:
### 1. Define the tSZ frequency scaling function and Taylor expansion ###

def get_physics_constants(nu):
    """Calculate the physics constants needed for tSZ scaling."""
    T_cmb = 2.7255     # CMB temperature in K
    k_B = 1.380649e-23 # Boltzmann constant in J/K
    h = 6.62607015e-34 # Planck constant in J*s
    x = (h * nu * 1e9) / (k_B * T_cmb) # Dimensionless frequency
    return x

def get_tsz_g(nu):
    """Calculate the tSZ frequency scaling g(nu) in dimensionless units."""
    x = get_physics_constants(nu)
    g_nu = x * (np.exp(x) + 1) / (np.exp(x) - 1) - 4 # or alternatively: g_nu = x * (1 / np.tanh(x / 2)) - 4
    return g_nu

def get_tsz_f(nu):
    """Calculate the tSZ frequency scaling f(nu) in dimensionless units."""
    x = get_physics_constants(nu)
    f_nu = (-x/2) * ( 1 / ( (np.exp(x) - np.exp(-x)) / 2 )**2 ) # or alternatively: f_nu = (-x/2) * (1 / np.sinh(x/2)**2)
    #f_nu = (-x/2) * (1 / np.sinh(x/2)**2)
    #f_nu_T_CMB = f_nu * T_cmb # IF DATA IN MU_K, CONVERT TO K
    return f_nu

In [ ]:
### 2. Load the processed Maps and the data_cube ###

path_to_websky = '/Users/joanribot/HEAVY_STUFF/TFM_data/WebSky_CMB_Mocks/'

# 1. Load the Data Cube (The 'Observed' data)
data_cube = hp.read_map(path_to_websky + "websky_data_cube_9freq.fits", field=None).astype(np.float32)

# 2. Load the Truth Maps (For validation and residuals)
y_map_truth = hp.read_map(path_to_websky + "websky_truth_y_map.fits").astype(np.float32)
cmb_map_truth = hp.read_map(path_to_websky + "websky_truth_cmb_map.fits").astype(np.float32)

# 3. Define your constants
frequencies = [30, 44, 70, 70, 100, 143, 217, 353, 545, 857]
T_cmb_muK = 2.7255e6

# Mixing vectors
g_tsz = np.array([get_tsz_g(nu) for nu in frequencies])
a_cmb = np.ones(len(frequencies)) # Since maps are in muK